# Yuva Intern Task 2 - Hands-On Data Lab Implementation

**Dataset note:** synthetic practice data, not real student records. Run cells from top to bottom.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
DATA = ROOT / "data" / "student_performance_raw.csv"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
df = pd.read_csv(DATA)
print("Rows and columns:", df.shape)
display(df.head())


## 1. Inspect data quality

In [ ]:
print("Data types:\n", df.dtypes)
print("\nMissing values:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())


## 2. Clean and transform

In [ ]:
df = df.drop_duplicates().copy()
numeric_cols = ["Age", "Study_Hours", "Attendance_Percent", "Math_Score", "Science_Score", "English_Score"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())
df["Attendance_Percent"] = df["Attendance_Percent"].clip(0, 100)
for col in ["Math_Score", "Science_Score", "English_Score"]:
    df[col] = df[col].clip(0, 100)
df["Average_Score"] = df[["Math_Score", "Science_Score", "English_Score"]].mean(axis=1).round(2)
df["Performance_Level"] = np.select([df["Average_Score"] >= 85, df["Average_Score"] >= 70],
                                    ["Excellent", "Good"], default="Needs Improvement")
df.to_csv(OUT / "student_performance_cleaned.csv", index=False)
print("Cleaned shape:", df.shape)
display(df.head())


## 3. Descriptive statistics and aggregation

In [ ]:
display(df[numeric_cols + ["Average_Score"]].describe().round(2))
summary = df.groupby("Performance_Level").agg(
    Student_Count=("Student_ID", "count"),
    Mean_Average_Score=("Average_Score", "mean"),
    Mean_Attendance=("Attendance_Percent", "mean")
).round(2)
display(summary)
summary.to_csv(OUT / "performance_summary.csv")


## 4. Visualizations

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(8,5))
sns.histplot(df["Average_Score"], bins=8, kde=True)
plt.title("Distribution of Student Average Scores")
plt.tight_layout()
plt.savefig(OUT / "01_score_distribution.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=df, x="Performance_Level", y="Average_Score",
            order=["Needs Improvement", "Good", "Excellent"], errorbar=None)
plt.title("Average Score by Performance Level")
plt.tight_layout()
plt.savefig(OUT / "02_performance_level.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x="Study_Hours", y="Average_Score", hue="Performance_Level")
plt.title("Study Hours vs Average Score")
plt.tight_layout()
plt.savefig(OUT / "03_study_hours_vs_score.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
corr = df[["Study_Hours", "Attendance_Percent", "Math_Score", "Science_Score", "English_Score", "Average_Score"]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig(OUT / "04_correlation_heatmap.png", dpi=150)
plt.show()


In [ ]:
subject_means = df[["Math_Score", "Science_Score", "English_Score"]].mean().reset_index()
subject_means.columns = ["Subject", "Mean_Score"]
plt.figure(figsize=(8,5))
sns.barplot(data=subject_means, x="Subject", y="Mean_Score", errorbar=None)
plt.title("Mean Score by Subject")
plt.tight_layout()
plt.savefig(OUT / "05_subject_average.png", dpi=150)
plt.show()


## 5. Findings and conclusion

Review the tables and charts above. Describe only patterns visible in your own run. These are descriptive results from a synthetic dataset and do not prove that study time or attendance causes higher marks.